# AI text detection using Qwen3-Embedding-0.6B

Uploading model, it has to be already saved locally

In [1]:
from model_upload import load_model_from_cache


model = load_model_from_cache("Qwen/Qwen3-Embedding-0.6B", "/Users/roberto/Università/Deep learning")

/Users/roberto/Università/Deep learning/AI-text-detection-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4159.01it/s]


Model uploaded!
Device modello: cpu


## Creating and saving embeddings

In [7]:
df = pd.read_parquet(PATH_TRAIN, columns=['text', 'generated']).head(NUM_SAMPLES)
print(df['text'].str.len().describe())  # lunghezza in caratteri

count      10.000000
mean     1741.800000
std      1705.668119
min        59.000000
25%       973.500000
50%      1126.000000
75%      1516.500000
max      5103.000000
Name: text, dtype: float64


In [2]:
import os
import torch
import pandas as pd

# ── IMPOSTAZIONI DI CONTROLLO ──────────────────────────────────────────────
MODALITA_TEST = True  

NUM_SAMPLES = 10 if MODALITA_TEST else 20000  

# File di output degli embedding
TRAIN_FILE = 'test_train_embeddings.pt' if MODALITA_TEST else 'train_embeddings.pt'
VAL_FILE   = 'test_val_embeddings.pt'   if MODALITA_TEST else 'val_embeddings.pt'
TEST_FILE  = 'test_test_embeddings.pt'  if MODALITA_TEST else 'test_embeddings.pt'

# Percorsi dei file parquet scaricati sul tuo Mac
PATH_TRAIN = "data/train-00000-of-00003.parquet"
PATH_VAL   = "data/validation-00000-of-00001.parquet"
PATH_TEST  = "data/test-00000-of-00001.parquet"

# ── FUNZIONE DI ELABORAZIONE LOCALE CORRETTA ───────────────────────────────
def process_local_parquet(file_path, num_samples, output_filename, split_name):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Non trovo il file {file_path}. Assicurati di averlo scaricato.")
        
    print(f"\n--- Elaborazione split locale: {split_name} ---")
    
    # CORREZIONE: usiamo 'generated' al posto di 'label' come richiesto dal file parquet
    df = pd.read_parquet(file_path, columns=['text', 'generated'], engine='pyarrow').head(num_samples)
    
    sentences = df['text'].tolist()
    labels = df['generated'].tolist()  # Estratta correttamente
    
    print(f"Campioni caricati: {len(sentences)}")
    print(f"Generazione embedding con Qwen per {split_name}...")
    
    # Calcolo effettivo degli embedding con Qwen
    X_embeddings = model.encode(
    sentences, 
    batch_size=1,  # <--- FORZA L'ELABORAZIONE DI 1 FRASE ALLA VOLTA
    convert_to_tensor=True, 
    show_progress_bar=True
)
    y_labels = torch.tensor(labels, dtype=torch.long)
    
    # Salvataggio permanente (manteniamo la chiave 'labels' così il codice della tua rete non cambia)
    torch.save({
        'embeddings': X_embeddings,
        'labels': y_labels
    }, output_filename)
    print(f"Salvato con successo in '{output_filename}' | Shape: {X_embeddings.shape}")

# ── ESECUZIONE ─────────────────────────────────────────────────────────────
process_local_parquet(PATH_TRAIN, NUM_SAMPLES, TRAIN_FILE, "Train")
process_local_parquet(PATH_VAL, NUM_SAMPLES, VAL_FILE, "Validation")
process_local_parquet(PATH_TEST, NUM_SAMPLES, TEST_FILE, "Test")

print("\n[FINISH] Errore risolto! Tutti i file .pt di test sono pronti.")


--- Elaborazione split locale: Train ---
Campioni caricati: 10
Generazione embedding con Qwen per Train...


Batches:   0%|          | 0/10 [00:50<?, ?it/s]


KeyboardInterrupt: 

In [20]:
sentences = [
    "Image caption The attack happened in Daburah, about 50 miles from the border with Bangladesh A woman and her three siblings have been arrested in Saudi Arabia after a horrific rape and murder in which they were found tied up and dead with their throats slit, the country's interior ministry said. Razan, 14, Nourat, 12, and Marwa, 10, were found hanged in Daburah in the south of the country during a dawn raid, it said. The victim and victims father were detained in the village following the discovery, according to the ministry. The family was reported to have close ties with local Islamists. It followed an initial arrest in the city of Jeddah two weeks ago, the interior ministry said in a statement on Friday. There has not been any comment from the suspects so far. The five-month-old girl was found naked, bleeding from her neck, with her stomach cut, according to local media reports. Her father and mother have been arrested as well for not bringing her to safety and not registering the incident immediately. Three of the suspects are juveniles. One has been found fit to stand trial. All five are being held at the Riyadh police headquarters. The five-year-old girl was sexually assaulted and killed by an Iranian man she had hired for companionship."
]    
embeddings = model.encode(sentences)
print(embeddings.shape)
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

(1, 1024)
torch.Size([1, 1])


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 1. Carica gli embedding pre-calcolati dal disco
train_data = torch.load('train_dataset_embeddings.pt')
X_train = train_data['embeddings']
y_train = train_data['labels']

# 2. Crea il DataLoader di PyTorch per il training a batch
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. Definisci la tua rete Fully Connected
# Qwen-0.6B genera embedding, la dimensione di input dipenderà da quella del modello (es. 1536 o 4096)
input_dim = X_train.shape[1] 

class TextClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TextClassifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2) # 2 classi: Human (0) e AI (1)
        )
        
    def forward(self, x):
        return self.fc(x)

model_fc = TextClassifier(input_dim)

# Ora puoi avviare il loop di training (le epoche gireranno in pochissimi millisecondi!)
# for epoch in range(epochs): ...